In [10]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from utilities import train_model
import numpy as np

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
EPOCHS = 128
BATCH_SIZE = 32

# Load training dataset into a single batch to compute mean and stddev.
transform = transforms.Compose([transforms.ToTensor()])
trainset = CIFAR10(root='./pt_data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=len(trainset), shuffle=False)
data = next(iter(trainloader))
mean = data[0].mean()
stddev = data[0].std()

# Load and standardize training and test dataset.
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(mean, stddev)])

trainset = CIFAR10(root='./pt_data', train=True, download=True, transform=transform)
testset = CIFAR10(root='./pt_data', train=False, download=True, transform=transform)

In [11]:
# Model with 2 convolutional and 1 fully-connected layer.
model = nn.Sequential(
    nn.Conv2d(3, 64, 5, stride=2, padding=2), # Output is 64x16x16.
    nn.ReLU(),
    nn.Conv2d(64, 64, 3, stride=2, padding=1), # Output is 64x8x8.
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10)
)

# Retrieve layers for custom weight initialization.
layers = next(model.modules())
conv_layer0 = layers[0]
conv_layer1 = layers[2]
output_layer = layers[5]

# Kaiming (He) initialization.
nn.init.kaiming_normal_(conv_layer0.weight)
nn.init.constant_(conv_layer0.bias, 0.0)
nn.init.kaiming_normal_(conv_layer1.weight)
nn.init.constant_(conv_layer1.bias, 0.0)

# Xavier (Glorot) initialization.
nn.init.xavier_uniform_(output_layer.weight)
nn.init.constant_(output_layer.bias, 0.0)

# Loss function and optimizer
optimizer = torch.optim.Adam(model.parameters())
loss_function = nn.CrossEntropyLoss()


We are now ready to train the network by calling the same train_model() function that was used in the c6e1_boston example, but this time we give the argument "acc" because we work on a classification problem and therefore want the function to compute accuracy instead of mean absolute error (mae).


In [12]:
# Train the model.
train_model(model, device, EPOCHS, BATCH_SIZE, trainset, testset,
            optimizer, loss_function, 'acc')


Epoch 1/128 loss: 1.3909 - acc: 0.5075 - val_loss: 1.1630 - val_acc: 0.5905
Epoch 2/128 loss: 1.0470 - acc: 0.6337 - val_loss: 1.0803 - val_acc: 0.6204
Epoch 3/128 loss: 0.9063 - acc: 0.6830 - val_loss: 1.0426 - val_acc: 0.6397
Epoch 4/128 loss: 0.8063 - acc: 0.7191 - val_loss: 1.0621 - val_acc: 0.6387
Epoch 5/128 loss: 0.7246 - acc: 0.7463 - val_loss: 1.1061 - val_acc: 0.6393
Epoch 6/128 loss: 0.6547 - acc: 0.7716 - val_loss: 1.1314 - val_acc: 0.6394
Epoch 7/128 loss: 0.5950 - acc: 0.7916 - val_loss: 1.2341 - val_acc: 0.6272
Epoch 8/128 loss: 0.5430 - acc: 0.8093 - val_loss: 1.2654 - val_acc: 0.6283
Epoch 9/128 loss: 0.4968 - acc: 0.8254 - val_loss: 1.3153 - val_acc: 0.6300
Epoch 10/128 loss: 0.4520 - acc: 0.8402 - val_loss: 1.3676 - val_acc: 0.6343
Epoch 11/128 loss: 0.4217 - acc: 0.8496 - val_loss: 1.4514 - val_acc: 0.6290
Epoch 12/128 loss: 0.3771 - acc: 0.8670 - val_loss: 1.5483 - val_acc: 0.6270
Epoch 13/128 loss: 0.3546 - acc: 0.8741 - val_loss: 1.6651 - val_acc: 0.6261
Epoch 14

[0.9766074856046065, 0.6101238019169329]